In [1]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

# Load the processed dataset
PROCESSED_PATH = os.path.join("..", "data", "processed", "imdb_clean.csv")
df = pd.read_csv(PROCESSED_PATH)

print(f"Loaded {len(df):,} cleaned reviews.")
df.head(3)

Loaded 50,000 cleaned reviews.


,review,clean_review,sentiment
0,One of the other reviewers has mentioned that ...,one reviewers mentioned watching oz episode ho...,positive
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...,positive
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...,positive


In [2]:
# Map text sentiments to binary integers
sentiment_map = {'positive': 1, 'negative': 0}
df['label'] = df['sentiment'].map(sentiment_map)

# Verify the mapping applied correctly (checking for any NaNs introduced)
missing_labels = df['label'].isnull().sum()
print(f"Missing labels after mapping: {missing_labels}")

# Display the mapping side-by-side to confirm
print("\n--- Mapping Verification ---")
print(df[['sentiment', 'label']].value_counts())

Missing labels after mapping: 0

--- Mapping Verification ---
sentiment  label
positive   1        25000
negative   0        25000
Name: count, dtype: int64


In [3]:
# Define features and target
X = df['clean_review']
y = df['label']

# 80/20 split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    stratify=y, 
    random_state=42
)

In [4]:
# 1. Check matrix dimensions
print("--- Split Shapes ---")
print(f"X_train : {X_train.shape[0]:,} rows")
print(f"X_test  : {X_test.shape[0]:,} rows")
print(f"y_train : {y_train.shape[0]:,} labels")
print(f"y_test  : {y_test.shape[0]:,} labels\n")

# 2. Verify stratified class distribution
print("--- Class Distribution (y_train) ---")
print(y_train.value_counts(normalize=True).round(4) * 100)

print("\n--- Class Distribution (y_test) ---")
print(y_test.value_counts(normalize=True).round(4) * 100)

--- Split Shapes ---
X_train : 40,000 rows
X_test  : 10,000 rows
y_train : 40,000 labels
y_test  : 10,000 labels

--- Class Distribution (y_train) ---
label
1    50.0
0    50.0
Name: proportion, dtype: float64

--- Class Distribution (y_test) ---
label
0    50.0
1    50.0
Name: proportion, dtype: float64


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Configure TF-IDF Vectorizer with deliberate parameters
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),     # Unigrams + Bigrams (captures negations like "not good", "highly recommend")
    sublinear_tf=True,      # Uses 1 + log(tf) scaling to dampen high-frequency term domination
    max_features=10000,     # Cap top features to balance accuracy with memory/speed
    min_df=2,               # Ignore ultra-rare terms that appear in fewer than 2 documents
    strip_accents='unicode'
)

print("TF-IDF Vectorizer initialized successfully.")

TF-IDF Vectorizer initialized successfully.


In [7]:
# 1. Fit vectorizer on training text ONLY, then transform training features
X_train_tfidf = vectorizer.fit_transform(X_train)

# 2. Transform test features using the already-fitted vectorizer
X_test_tfidf = vectorizer.transform(X_test)

print("--- Matrix Dimensions ---")
print(f"X_train_tfidf : {X_train_tfidf.shape[0]:,} samples × {X_train_tfidf.shape[1]:,} features")
print(f"X_test_tfidf  : {X_test_tfidf.shape[0]:,} samples × {X_test_tfidf.shape[1]:,} features")

--- Matrix Dimensions ---
X_train_tfidf : 40,000 samples × 10,000 features
X_test_tfidf  : 10,000 samples × 10,000 features


In [8]:
# Inspect learned vocabulary and feature names
feature_names = vectorizer.get_feature_names_out()

print(f"Vocabulary Size: {len(feature_names):,} n-grams\n")

print("--- Sample Unigrams & Bigrams from Vocabulary ---")
# Show a mixture of single words and two-word phrases
sample_features = list(feature_names[::500])  # Sample every 500th feature
for feat in sample_features[:15]:
    print(f"  • {feat}")

Vocabulary Size: 10,000 n-grams

--- Sample Unigrams & Bigrams from Vocabulary ---
  • aaron
  • aspiring
  • breaking
  • club
  • david lynch
  • effective
  • fay
  • funny movie
  • heads
  • ironic
  • likeable
  • mgm
  • neglected
  • patricia
  • pursuit


In [9]:
# Handle any potential NaNs in text columns
X_train = X_train.fillna("")
X_test = X_test.fillna("")

# 1. Fit ONLY on X_train, then transform X_train into a sparse matrix
X_train_tfidf = vectorizer.fit_transform(X_train)

# 2. Transform X_test using the already-fitted vectorizer (NEVER fit on X_test)
X_test_tfidf = vectorizer.transform(X_test)

# 3. Rigorous Sanity Check Assertions
assert X_train_tfidf.shape[1] == X_test_tfidf.shape[1], (
    f"❌ Mismatch in feature count! Train features: {X_train_tfidf.shape[1]}, Test features: {X_test_tfidf.shape[1]}"
)
assert X_train_tfidf.shape[0] == len(X_train), (
    f"❌ Train row count mismatch! Matrix: {X_train_tfidf.shape[0]}, Original: {len(X_train)}"
)
assert X_test_tfidf.shape[0] == len(X_test), (
    f"❌ Test row count mismatch! Matrix: {X_test_tfidf.shape[0]}, Original: {len(X_test)}"
)

print("✅ All transformation pipeline assertions passed!")
print(f"• Training Matrix : {X_train_tfidf.shape[0]:,} samples × {X_train_tfidf.shape[1]:,} features")
print(f"• Testing Matrix  : {X_test_tfidf.shape[0]:,} samples × {X_test_tfidf.shape[1]:,} features")

✅ All transformation pipeline assertions passed!
• Training Matrix : 40,000 samples × 10,000 features
• Testing Matrix  : 10,000 samples × 10,000 features


In [10]:
import joblib
import json

# Ensure models directory exists
MODELS_DIR = os.path.join("..", "models")
os.makedirs(MODELS_DIR, exist_ok=True)

# 1. Serialize the fitted vectorizer artifact for API and inference usage
vectorizer_path = os.path.join(MODELS_DIR, "tfidf_vectorizer.joblib")
joblib.dump(vectorizer, vectorizer_path)
print(f"✅ Fitted vectorizer artifact saved to: {vectorizer_path}")

# 2. Extract configuration decisions for documentation
config_summary = {
    "ngram_range": vectorizer.ngram_range,
    "max_features": vectorizer.max_features,
    "sublinear_tf": vectorizer.sublinear_tf,
    "min_df": vectorizer.min_df,
    "strip_accents": vectorizer.strip_accents,
    "vocabulary_size": len(vectorizer.vocabulary_)
}

print("\n--- Vectorizer Configuration Summary (Copy to README.md) ---")
for key, value in config_summary.items():
    print(f"  • {key:<20}: {value}")

✅ Fitted vectorizer artifact saved to: ..\models\tfidf_vectorizer.joblib

--- Vectorizer Configuration Summary (Copy to README.md) ---
  • ngram_range         : (1, 2)
  • max_features        : 10000
  • sublinear_tf        : True
  • min_df              : 2
  • strip_accents       : unicode
  • vocabulary_size     : 10000


In [11]:
# Define paths for feature matrices and labels
train_features_path = os.path.join("..", "data", "processed", "X_train_tfidf.joblib")
test_features_path  = os.path.join("..", "data", "processed", "X_test_tfidf.joblib")
train_labels_path    = os.path.join("..", "data", "processed", "y_train.joblib")
test_labels_path     = os.path.join("..", "data", "processed", "y_test.joblib")

# Save artifacts
joblib.dump(X_train_tfidf, train_features_path)
joblib.dump(X_test_tfidf, test_features_path)
joblib.dump(y_train, train_labels_path)
joblib.dump(y_test, test_labels_path)

print("✅ Processed training/testing matrices saved to data/processed/")

✅ Processed training/testing matrices saved to data/processed/
